# Orthogonal Matching Pursuit with a Gaussian Bump Dictionary

**Orthogonal Matching Pursuit (OMP)** is a greedy algorithm for sparse signal approximation. Given a signal $y \in \mathbb{R}^n$ and an overcomplete dictionary $\Phi = [\phi_1 \mid \cdots \mid \phi_M] \in \mathbb{R}^{n \times M}$ (with $M \gg n$), OMP finds a sparse representation
$$
y \approx \Phi\,\alpha, \qquad \|\alpha\|_0 \ll M,
$$
by iteratively selecting the atom most correlated with the current residual, then re-projecting.

**Why it matters.** Many signals in signal processing, imaging, and statistics admit sparse representations in a suitable dictionary. Recovering those representations from noisy or compressively sampled observations is a central problem in compressed sensing and statistical learning.

**The OMP algorithm.** Starting from residual $r_0 = y$, at each step $k$:

1. **Atom selection** — find the dictionary column most correlated with the current residual:
$$
j_k = \arg\max_{j \notin S_{k-1}} \bigl|\langle r_{k-1},\, \phi_j \rangle\bigr|.
$$
2. **Support update** — $S_k = S_{k-1} \cup \{j_k\}$.
3. **Orthogonal projection** — solve the least-squares problem on the active subdictionary $\Phi_{S_k}$:
$$
\hat\alpha_{S_k} = \arg\min_{\alpha} \|y - \Phi_{S_k}\,\alpha\|^2 = (\Phi_{S_k}^\top \Phi_{S_k})^{-1}\Phi_{S_k}^\top y.
$$
4. **Residual update** — $r_k = y - \Phi_{S_k}\hat\alpha_{S_k}$.

**Convergence.** The residual norm decreases monotonically: $\|r_k\| \leq \|r_{k-1}\|$. Under suitable conditions (restricted isometry property of $\Phi$), the relative residual satisfies
$$
\frac{\|r_k\|}{\|y\|} \to 0 \quad \text{as } k \to \infty.
$$

**This notebook** uses a **Gaussian bump dictionary** at multiple scales and centres to reconstruct a moderately sparse smooth signal. We visualise: the dictionary atoms, the true and noisy signals, the step-by-step reconstruction progression, and the residual decay curve.

## Setup: imports and dictionary construction

We build a Gaussian bump dictionary on $n = 350$ uniformly spaced points in $[0,1]$. Each atom is a normalised Gaussian
$$
\phi_{c,\sigma}(t) \propto \exp\!\left(-\frac{(t-c)^2}{2\sigma^2}\right),
$$
with centres $c$ drawn from a fine grid and scales $\sigma \in \{0.018, 0.030, 0.050\}$. This multi-scale design allows the dictionary to represent both narrow and broad features. The resulting dictionary has $M = 270$ atoms.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True
rng = np.random.default_rng(42)

OUT = Path("python/orthogonal-matching-pursuit")
OUT.mkdir(parents=True, exist_ok=True)

# Signal domain
n = 350
t = np.linspace(0, 1, n)

# Dictionary: Gaussian bumps at multiple scales and centres
centers = np.linspace(0.03, 0.97, 90)
sigmas = np.array([0.018, 0.030, 0.050])
atoms = []
for s in sigmas:
    for c in centers:
        g = np.exp(-0.5 * ((t - c) / s) ** 2)
        g /= np.linalg.norm(g) + 1e-12
        atoms.append(g)
Phi = np.column_stack(atoms)   # shape (n, M), M = 270

print(f"Dictionary shape: {Phi.shape}  ({Phi.shape[1]} atoms, {Phi.shape[0]} samples)")

## Dictionary atoms

We display 16 representative atoms from the dictionary — a mix of scales and positions. The three scales produce qualitatively different atoms:

- **Narrow** ($\sigma \approx 0.018$): sharp, localised bumps that can resolve fine-scale features.
- **Medium** ($\sigma \approx 0.030$): intermediate width, balancing localisation and smoothness.
- **Wide** ($\sigma \approx 0.050$): broad bumps that capture slow, large-scale variations.

All atoms are $\ell^2$-normalised: $\|\phi_j\| = 1$.

In [ ]:
atoms_per_scale = len(centers)  # 90 per scale
scale_color = ["tab:blue", "tab:orange", "tab:green"]
scale_labels = {
    0: r"narrow ($\sigma\approx0.018$)",
    1: r"medium ($\sigma\approx0.030$)",
    2: r"wide  ($\sigma\approx0.050$)",
}

# Pick 6 + 5 + 5 = 16 atoms spread uniformly across each scale
selected_indices = []
picks_per_scale = [6, 5, 5]
offset = 0
for k_sc, cnt in enumerate(picks_per_scale):
    idx = np.round(np.linspace(0, atoms_per_scale - 1, cnt)).astype(int)
    selected_indices.extend((offset + idx).tolist())
    offset += atoms_per_scale

fig, axes = plt.subplots(4, 4, figsize=(11, 7), constrained_layout=True)
for panel_idx, atom_idx in enumerate(selected_indices):
    ax = axes[panel_idx // 4, panel_idx % 4]
    scale_id = atom_idx // atoms_per_scale
    ax.plot(t, Phi[:, atom_idx], color=scale_color[scale_id], lw=1.6)
    ax.set_ylim(-0.02, None)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"atom {atom_idx}", fontsize=8)

patches = [mpatches.Patch(color=scale_color[k], label=scale_labels[k]) for k in range(3)]
fig.legend(handles=patches, loc="lower center", ncol=3, fontsize=9,
           bbox_to_anchor=(0.5, -0.03))
fig.suptitle("Selected Gaussian bump dictionary atoms (16 of 270)", fontsize=12)

## Target signal: moderately sparse composition

We construct a target signal as a **linear combination of 7 dictionary atoms** — a moderately sparse signal — then add mild Gaussian noise:
$$
y_0 = \Phi\,\alpha^\star, \qquad y = y_0 + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma_{\mathrm{noise}}^2 I).
$$
With 7 active atoms (rather than 2 or 3), the reconstruction task is more challenging and the progression of the OMP approximation becomes more instructive to observe. The dashed vertical lines below mark the peak locations of the active atoms.

In [ ]:
# Ground-truth: 7 active atoms spanning different scales and positions
coef_true = np.zeros(Phi.shape[1])
idx_true = [10, 55, 95, 140, 170, 220, 255]
coef_true[idx_true] = [1.4, -1.0, 1.2, 0.8, -1.3, 0.9, -0.7]

y0 = Phi @ coef_true
noise_level = 0.03
y = y0 + noise_level * rng.normal(size=n)

print(f"True support size : {len(idx_true)} atoms")
print(f"Signal-to-noise   : {np.linalg.norm(y0) / (noise_level * np.sqrt(n)):.1f}")

fig, ax = plt.subplots(figsize=(11, 3.5), constrained_layout=True)
ax.plot(t, y, color="0.65", lw=1.2, label="observed $y$ (noisy)")
ax.plot(t, y0, "k", lw=1.8, label="true signal $y_0$")
for ii in idx_true:
    peak_loc = t[np.argmax(Phi[:, ii])]
    ax.axvline(peak_loc, color="tab:red", lw=0.9, ls="--", alpha=0.55)
ax.set_xlabel("$t$")
ax.set_title("True signal (7-sparse) and noisy observation")
ax.legend()

## OMP iterations: step-by-step reconstruction

We run OMP for $K = 20$ iterations and store the approximation $\hat y^{(k)} = \Phi_{S_k}\hat\alpha_{S_k}$ at each step. The relative residual is
$$
\rho_k = \frac{\|r_k\|}{\|y\|}.
$$
The complete iteration proceeds as follows:

- **Atom selection:** $j_k = \arg\max_j |\langle r_{k-1}, \phi_j \rangle|$
- **Orthogonal projection:** $\hat\alpha_{S_k} = (\Phi_{S_k}^\top \Phi_{S_k})^{-1} \Phi_{S_k}^\top y$
- **Residual update:** $r_k = y - \Phi_{S_k} \hat\alpha_{S_k}$

We then display snapshots at $k \in \{1, 2, 3, 4, 6, 8, 12, 20\}$ to see how the reconstruction improves step by step.

In [ ]:
K = 20
snapshots = [1, 2, 3, 4, 6, 8, 12, 20]

res = y.copy()
S = []
errs = []
y_hats = {}

for k in range(1, K + 1):
    corr = np.abs(Phi.T @ res)
    if S:
        corr[S] = -1.0
    j = int(np.argmax(corr))
    S.append(j)

    A = Phi[:, S]
    cS, *_ = np.linalg.lstsq(A, y, rcond=None)
    res = y - A @ cS
    errs.append(np.linalg.norm(res) / (np.linalg.norm(y) + 1e-12))

    if k in snapshots:
        coef_k = np.zeros(Phi.shape[1])
        coef_k[S] = cS
        y_hats[k] = Phi @ coef_k

S_final = list(S)
coef_final = np.zeros(Phi.shape[1])
cS_final, *_ = np.linalg.lstsq(Phi[:, S_final], y, rcond=None)
coef_final[S_final] = cS_final

print(f"Relative residual after {K} iterations: {errs[-1]:.4f}")

# Multi-panel progression figure
fig, axes = plt.subplots(2, 4, figsize=(14, 6), constrained_layout=True)
axes = axes.ravel()
for panel_idx, k in enumerate(snapshots):
    ax = axes[panel_idx]
    ax.plot(t, y0, color="0.7", lw=1.4, label="true")
    ax.plot(t, y_hats[k], color="tab:blue", lw=1.8, label=f"OMP $k={k}$")
    ax.set_title(f"$k = {k}$,  $\\rho = {errs[k-1]:.3f}$", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    if panel_idx == 0:
        ax.legend(fontsize=8, loc="upper right")
fig.suptitle("OMP reconstruction progression at selected iterations", fontsize=12)

## Final reconstruction, residual curve, and selected support

Three summary views are shown:

1. **Final approximation** $\hat y^{(K)}$ overlaid on the true and observed signals.
2. **Residual decay curve** $\rho_k = \|r_k\|/\|y\|$ versus iteration $k$ (semilog scale). Rapid early decay confirms that OMP greedily captures the highest-energy components first.
3. **Atom selection order** — the indices $j_k$ chosen by OMP at each iteration, compared with the true support (red dashed lines).

The orthogonal projection guarantees $\|r_k\| \leq \|r_{k-1}\|$ at every step, and the greedy strategy typically recovers the true support within a small multiple of its cardinality.

In [ ]:
fig = plt.figure(figsize=(13, 8), constrained_layout=True)
gs = gridspec.GridSpec(2, 2, figure=fig)

# Panel 1: final reconstruction
ax0 = fig.add_subplot(gs[0, :])
ax0.plot(t, y, color="0.65", lw=1.0, label="observed $y$")
ax0.plot(t, y0, "k", lw=1.6, label="true $y_0$")
ax0.plot(t, Phi @ coef_final, color="tab:blue", lw=2.0, label=f"OMP $K={K}$")
ax0.set_title(f"Final OMP reconstruction ($K={K}$ atoms)", fontsize=11)
ax0.legend()

# Panel 2: residual decay
ax1 = fig.add_subplot(gs[1, 0])
ax1.semilogy(range(1, K + 1), errs, "o-", color="tab:orange", lw=1.8, ms=5)
ax1.axhline(errs[-1], color="0.5", ls="--", lw=1)
for k in snapshots:
    ax1.axvline(k, color="tab:blue", lw=0.8, ls=":", alpha=0.6)
ax1.set_xlabel("iteration $k$")
ax1.set_ylabel(r"$\rho_k = \|r_k\|/\|y\|$")
ax1.set_title("Relative residual decay")
ax1.set_xticks(range(1, K + 1))

# Panel 3: atom selection order vs true support
ax2 = fig.add_subplot(gs[1, 1])
ax2.stem(range(1, K + 1), S_final, linefmt="tab:blue", markerfmt="o",
         basefmt="k", label="OMP selected atom")
for ii, true_idx in enumerate(idx_true):
    ax2.axhline(true_idx, color="tab:red", lw=1.0, ls="--", alpha=0.7,
                label="true support" if ii == 0 else None)
ax2.set_xlabel("iteration $k$")
ax2.set_ylabel("atom index $j$")
ax2.set_title("Atom selection order")
ax2.legend(fontsize=9)

fig.savefig(OUT / "snippet.png", bbox_inches="tight")

## Interactive exploration: OMP at $k$ iterations

Use the slider below to vary the number of OMP iterations $k \in [1, K]$ and observe how the reconstruction $\hat y^{(k)}$ evolves. At each step the orthogonal projection ensures the best possible fit within the current support $S_k$.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Pre-compute all K approximations
all_yhats = {}
res_w = y.copy()
S_w = []
for k in range(1, K + 1):
    corr_w = np.abs(Phi.T @ res_w)
    if S_w:
        corr_w[S_w] = -1.0
    j_w = int(np.argmax(corr_w))
    S_w.append(j_w)
    A_w = Phi[:, S_w]
    c_w, *_ = np.linalg.lstsq(A_w, y, rcond=None)
    res_w = y - A_w @ c_w
    cf = np.zeros(Phi.shape[1])
    cf[S_w] = c_w
    all_yhats[k] = Phi @ cf

slider = widgets.IntSlider(value=1, min=1, max=K, step=1,
                           description="$k$:", continuous_update=True)
out_w = widgets.Output()

def update(change):
    kk = change["new"]
    with out_w:
        out_w.clear_output(wait=True)
        fig_w, ax_w = plt.subplots(figsize=(11, 3.5))
        ax_w.plot(t, y, color="0.65", lw=1.0, label="observed")
        ax_w.plot(t, y0, "k", lw=1.6, label="true")
        ax_w.plot(t, all_yhats[kk], color="tab:blue", lw=2.0,
                  label=f"OMP k={kk},  rho={errs[kk-1]:.3f}")
        ax_w.legend()
        ax_w.set_title(f"OMP reconstruction at iteration k = {kk}")
        ax_w.grid(True)
        plt.tight_layout()
        plt.show()

slider.observe(update, names="value")
display(widgets.VBox([slider, out_w]))
update({"new": 1})

## Static snapshot

The cell below renders a single static frame of the interactive widget for automated execution contexts where `ipywidgets` are not displayed. Set `STATIC_SNAPSHOT = True` to enable it (default).

In [ ]:
STATIC_SNAPSHOT = True
if STATIC_SNAPSHOT:
    k_show = 8
    fig_s, ax_s = plt.subplots(figsize=(11, 3.5), constrained_layout=True)
    ax_s.plot(t, y, color="0.65", lw=1.0, label="observed")
    ax_s.plot(t, y0, "k", lw=1.6, label="true")
    ax_s.plot(t, y_hats[k_show], color="tab:blue", lw=2.0,
              label=f"OMP k={k_show},  rho={errs[k_show-1]:.3f}")
    ax_s.legend()
    ax_s.set_title(f"OMP reconstruction at iteration k = {k_show} (static snapshot)")

## Takeaways

- **OMP is a greedy pursuit**: at each step it selects the single atom most correlated with the current residual, then re-projects orthogonally onto the span of all selected atoms. This guarantees the residual is orthogonal to every previously chosen atom.
- **Rapid early convergence**: the first few iterations capture the dominant energy components, causing $\rho_k$ to drop sharply. Later iterations refine the approximation by recovering weaker components.
- **Support recovery**: under the restricted isometry property (RIP) of the dictionary, OMP exactly recovers a $s$-sparse signal in at most $s$ iterations. For highly coherent dictionaries (like overlapping Gaussian bumps), recovery is approximate but still high-quality.
- **Multi-scale dictionaries**: having atoms at multiple widths gives OMP the flexibility to represent features of very different spatial extents. The greedy selection naturally adapts to the dominant scale at each iteration.
- **Orthogonal vs plain matching pursuit**: the orthogonal projection at each step is the key difference from standard MP — it prevents redundant energy from being re-selected from already-chosen atoms and ensures monotone residual decay.

## Bibliographical Resources

- S. Mallat, Z. Zhang, "Matching Pursuits with Time-Frequency Dictionaries", *IEEE Transactions on Signal Processing*, 1993.
- Y. C. Pati, R. Rezaiifar, P. S. Krishnaprasad, "Orthogonal Matching Pursuit: Recursive Function Approximation with Applications to Wavelet Decomposition", *Asilomar Conference on Signals, Systems and Computers*, 1993.
- J. A. Tropp, A. C. Gilbert, "Signal Recovery from Random Measurements via Orthogonal Matching Pursuit", *IEEE Transactions on Information Theory*, 2007.
- E. J. Cand\u00e8s, M. B. Wakin, "An Introduction to Compressive Sampling", *IEEE Signal Processing Magazine*, 2008.
- T. Hastie, R. Tibshirani, M. Wainwright, *Statistical Learning with Sparsity: The Lasso and Generalizations*, CRC Press, 2015.